# 03 - Cleaning & Normalization Pipeline (Task 4)
**Owner:** Rabi Sandhu

All code is in `src/clean.py` -> one function `clean_text()`.

In [ ]:
import sys, re
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
df = pd.read_csv(ROOT / "data/processed/sample_raw.csv")
print(df.shape)

In [ ]:
import clean
from clean import clean_text, normalize_urdu, remove_noise, light_stem, URDU_STOPWORDS, NEGATIONS

## Step-by-step on one raw-style example

In [ ]:
raw = ("اسلام آباد: عالمی بینک نے ۲۰ لاکھ ڈالر فراہم کرنے کا اعلان کیا، "
       "جس سے متاثرہ خاندانوں کی مدد ہوگی۔ برآمدات میں اضافہ نہیں ہوا۔ <p>Dawn</p> https://dawn.com كتاب")
s1 = normalize_urdu(raw);          print("1-2 NFC + Urdu norm :", s1)
s2 = remove_noise(s1);             print("3   noise removed   :", s2)
s3 = clean.NUMBER.sub(" NUM ", s2.lower()); print("4-5 case + numbers  :", s3)
t = s3.split();                    print("6   tokens          :", t)
t = [w for w in t if w not in URDU_STOPWORDS]; print("7   no stop words   :", t)
t = [light_stem(w) for w in t];    print("8   stemmed         :", t)
print("\nclean_text() gives  :", clean_text(raw))

## Urdu normalization examples

In [ ]:
for a in ["كتاب", "يہ", "۲۰۲۴", "رکھتی ہےفنانس", "متاثر ہیںیہ بھی"]:
    print(repr(a), "->", repr(normalize_urdu(a)))

**Glued words fix:** in Urdu, `ے` and `ں` only come at the END of a word. If a letter comes right after them, two words were glued
(because the dataset authors removed the full stops). So we add a space there: `ہےفنانس -> ہے فنانس`. This fixes most glued sentences.

In [ ]:
import re
docs_raw = df["News Text"].tolist()
before = sum(len(clean.GLUED.findall(d)) for d in docs_raw)
print("Glued words found and split in the sample:", before)

## Stop words - negations are kept

In [ ]:
print("Stop words:", len(URDU_STOPWORDS))
print("Negations kept:", NEGATIONS, "| any in stop list?", bool(NEGATIONS & URDU_STOPWORDS))
print(clean_text("قیمتوں میں اضافہ ہوا"), "  vs  ", clean_text("قیمتوں میں اضافہ نہیں ہوا"))

## Stemming examples (light suffix stemmer)

In [ ]:
words = ["خاندانوں", "شہریوں", "کمپنیاں", "کتابیں", "ملازمین", "منصوبوں", "معاملات", "خواتین", "باتیں"]
pd.DataFrame({"word": words, "stem": [light_stem(w) for w in words]})

**Why stemming and not lemmatization:** NLTK and spaCy have no Urdu lemmatizer (it needs a dictionary + POS tags).
A light stemmer merges plural forms (`خاندانوں -> خاندان`, `کمپنیاں -> کمپنی`) and is enough for topic classification.
Downside: some words are over-stemmed (`منصوبوں -> منصوب`, `معاملات -> معامل`) and Arabic plurals (`خواتین`, `ملازمین`) are not handled.

## Vocabulary before vs after (whole sample)

In [ ]:
docs = (df["Headline"].fillna("") + " " + df["News Text"].fillna("")).tolist()
cleaned = [clean_text(d) for d in docs]
vb = Counter(w for d in docs for w in d.split()); va = Counter(w for d in cleaned for w in d.split())
pd.DataFrame({"tokens": [sum(vb.values()), sum(va.values())], "types (vocab)": [len(vb), len(va)]},
             index=["before", "after"])

In [ ]:
for d, c in list(zip(docs, cleaned))[:3]:
    print("BEFORE:", d[:150]); print("AFTER :", c[:150]); print()